# 04 — Full Building Facade Analysis Pipeline

This notebook demonstrates the **end-to-end building facade analysis pipeline** from the `building_analyzer` repository.

The pipeline chains three ML modules in sequence:
1. **Facade Segmentation** — pixel-wise classification of structural elements
2. **Damage Detection** — bounding-box localisation of damage instances
3. **Material Classification** — material identification in each region

The final output is a structured `BuildingAnalysisResult` JSON report.


In [ ]:
!pip install torch torchvision albumentations Pillow numpy matplotlib fastapi uvicorn python-multipart httpx --quiet

In [ ]:
import subprocess, sys, os
if not os.path.exists('building_analyzer'):
    subprocess.run(['git', 'clone', 'https://github.com/Tripoid/building_analyzer.git'], check=True)
REPO_ROOT = os.path.abspath('building_analyzer')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('Ready. REPO_ROOT =', REPO_ROOT)

In [ ]:
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from ml.facade_segmentation.dataset  import FACADE_CLASS_NAMES
from ml.facade_segmentation.model    import UNetSegmentation
from ml.facade_segmentation.inference import SegmentationInferencer, SegmentationInferencerConfig
from ml.damage_detection.dataset     import DAMAGE_CLASS_NAMES
from ml.damage_detection.model       import AnchorFreeDamageDetector
from ml.damage_detection.inference   import DamageDetectionInferencer, DamageDetectionInferencerConfig
from ml.material_classification.dataset import MATERIAL_CLASS_NAMES
from ml.material_classification.model   import CNNMaterialClassifier
from ml.material_classification.inference import MaterialInferencer, MaterialInferencerConfig
from ml.pipeline.pipeline import (
    BuildingAnalysisPipeline, PipelineConfig,
    SegmentationStageConfig, DamageStageConfig, MaterialStageConfig
)
from ml.pipeline.result import BuildingAnalysisResult

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Construct the pipeline with lightweight models

In [ ]:
# --- Segmentation inferencer ------------------------------------------------
seg_model = UNetSegmentation(
    num_classes=len(FACADE_CLASS_NAMES),
    class_names=FACADE_CLASS_NAMES,
    base_channels=16,
)
seg_inferencer = SegmentationInferencer(
    model=seg_model,
    config=SegmentationInferencerConfig(device=DEVICE, image_size=(128, 128)),
)

# --- Damage detection inferencer --------------------------------------------
det_model = AnchorFreeDamageDetector(
    num_classes=len(DAMAGE_CLASS_NAMES),
    class_names=DAMAGE_CLASS_NAMES,
    fpn_channels=64,
)
det_inferencer = DamageDetectionInferencer(
    model=det_model,
    config=DamageDetectionInferencerConfig(device=DEVICE, image_size=(128, 128), score_threshold=0.3),
)

# --- Material classification inferencer ------------------------------------
mat_model = CNNMaterialClassifier(
    num_classes=len(MATERIAL_CLASS_NAMES),
    class_names=MATERIAL_CLASS_NAMES,
    base_channels=16,
)
mat_inferencer = MaterialInferencer(
    model=mat_model,
    config=MaterialInferencerConfig(device=DEVICE, image_size=(64, 64)),
)

# --- Full pipeline ----------------------------------------------------------
pipeline_config = PipelineConfig(
    device=DEVICE,
    segmentation=SegmentationStageConfig(enabled=True),
    damage=DamageStageConfig(enabled=True, score_threshold=0.3),
    materials=MaterialStageConfig(enabled=True),
)
pipeline = BuildingAnalysisPipeline(
    config=pipeline_config,
    seg_model=seg_inferencer,
    det_model=det_inferencer,
    mat_model=mat_inferencer,
)
print('Pipeline ready!')

## 2. Run the pipeline on a synthetic test image

In [ ]:
# Create a synthetic test image (replace with a real facade photo in practice)
test_image = np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8)

print('Running full pipeline analysis...')
result: BuildingAnalysisResult = pipeline.analyze(test_image)
print(f'Done in {result.metadata["elapsed_seconds"]:.2f}s')
print(f'Damage instances detected: {result.num_damage_instances}')

## 3. Inspect the structured result

In [ ]:
print('=== SEGMENTATION SUMMARY ===')
seg = result.segmentation
print(f'Dominant class:     {seg.dominant_class}')
print(f'Damaged fraction:   {seg.damaged_area_fraction:.3f}')
print('Area fractions:')
for cls, frac in sorted(seg.class_area_fractions.items(), key=lambda x: -x[1]):
    bar = '█' * int(frac * 40)
    print(f'  {cls:12s}: {frac:.3f}  {bar}')

In [ ]:
print('=== DAMAGE INSTANCES ===')
for i, di in enumerate(result.damage_instances[:10]):
    print(f'  [{i+1}] {di.label_name}  score={di.score:.3f}  '
          f'box=[{di.box[0]:.0f},{di.box[1]:.0f},{di.box[2]:.0f},{di.box[3]:.0f}]  '
          f'material={di.material_in_region}')
if result.num_damage_instances > 10:
    print(f'  ... and {result.num_damage_instances - 10} more')

In [ ]:
print('=== MATERIAL SUMMARY ===')
mat = result.materials
print(f'Overall dominant:  {mat.overall_dominant_material}')
print(f'Intact regions:    {mat.intact_material}')
print(f'Damaged regions:   {mat.damaged_material}')

## 4. JSON export

In [ ]:
result_dict = result.to_dict()
# Truncate damage_instances for display
display_dict = dict(result_dict)
display_dict['damage_instances'] = display_dict['damage_instances'][:3]
print(json.dumps(display_dict, indent=2))

## 5. Visualise stages

In [ ]:
# Stage 1: segmentation mask
seg_pred = seg_inferencer.predict_from_array(test_image)
seg_overlay = seg_pred.overlay_on(test_image, alpha=0.5)

# Stage 2: damage detection
det_pred = det_inferencer.predict_from_array(test_image)
det_vis  = det_pred.visualize(test_image)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(test_image);    axes[0].set_title('Input Image');           axes[0].axis('off')
axes[1].imshow(seg_overlay);   axes[1].set_title('Segmentation Overlay');  axes[1].axis('off')
axes[2].imshow(det_vis);       axes[2].set_title('Damage Detection');       axes[2].axis('off')
plt.suptitle('Building Facade Analysis — Stage Outputs', fontsize=14)
plt.tight_layout(); plt.show()

## 6. Batch analysis

In [ ]:
# Analyse multiple images at once
batch = [np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8) for _ in range(4)]
batch_results = pipeline.analyze_batch(batch)

print(f'Analysed {len(batch_results)} images.')
for i, r in enumerate(batch_results):
    print(f'  Image {i+1}: dominant_class={r.segmentation.dominant_class}  '
          f'damage={r.num_damage_instances}  material={r.materials.intact_material}')

## 7. Disabling individual stages

Any stage can be disabled without changing the rest of the pipeline:

In [ ]:
# Run only segmentation (disable damage detection and material classification)
pipeline_seg_only = BuildingAnalysisPipeline(
    config=PipelineConfig(
        device=DEVICE,
        segmentation=SegmentationStageConfig(enabled=True),
        damage=DamageStageConfig(enabled=False),
        materials=MaterialStageConfig(enabled=False),
    ),
    seg_model=seg_inferencer,
    det_model=det_inferencer,
    mat_model=mat_inferencer,
)
result_seg_only = pipeline_seg_only.analyze(test_image)
print('Segmentation-only result:')
print(f'  dominant_class: {result_seg_only.segmentation.dominant_class}')
print(f'  damage_instances: {result_seg_only.num_damage_instances}')
print(f'  materials: {result_seg_only.materials.intact_material}')

## 8. Swapping models

The registry pattern makes it trivial to swap any module's model:

In [ ]:
from ml.common.registry import ModelRegistry
from ml.facade_segmentation.model import DeepLabV3PlusSegmentation

# Build a DeepLabV3+ model via the registry
new_seg_model = ModelRegistry.build(
    'deeplabv3plus',
    namespace='segmentation',
    num_classes=len(FACADE_CLASS_NAMES),
    class_names=FACADE_CLASS_NAMES,
    encoder_name='simple',
)
new_seg_inferencer = SegmentationInferencer(
    model=new_seg_model,
    config=SegmentationInferencerConfig(device=DEVICE, image_size=(128, 128)),
)

# Drop in the new inferencer — everything else stays the same
pipeline_v2 = BuildingAnalysisPipeline(
    config=pipeline_config,
    seg_model=new_seg_inferencer,
    det_model=det_inferencer,
    mat_model=mat_inferencer,
)
result_v2 = pipeline_v2.analyze(test_image)
print(f'DeepLabV3+ pipeline result: {result_v2.segmentation.dominant_class}')